### [한번 해보기] 영화 추천 시스템

##### 1. 데이터셋 로드

In [3]:
import pandas as pd

df = pd.read_csv('./data/raw/tmdb_5000_movies.csv')

##### 2. ChromaDB Collection 생성

In [4]:
import chromadb

movie_client = chromadb.PersistentClient(path='./db/movie_chroma_db')
movie_collection = movie_client.get_or_create_collection(name='movie_documents')

##### 3. Collection.add

In [5]:
from sentence_transformers import SentenceTransformer

movie_embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
for i in range(len(df)):
    overview_embed = movie_embedding_model.encode(str(df['overview'][i])).tolist()

    movie_collection.add(
        ids=[str(df['id'][i])],
        embeddings=[overview_embed],
        metadatas=[{'title': df['title'][i], 'overview_text': df['overview'][i]}]
)

##### 4. 유사도 가빈 영화 추천

In [7]:
# 1) [제목] 입력 -> 줄거리를 찾아서 -> 줄거리로 유사도 검색 -> [영화] 추천
input_title = 'Avatar'
input_overview = df[df['title'] == input_title]['overview']
query_embed = movie_embedding_model.encode(input_overview).tolist()

top_n = 5

results = movie_collection.query(query_embeddings=query_embed, n_results=top_n)

for i, result in enumerate(results['metadatas'][0]):
    print(f'| {i+1} | Title: [ {result['title']} ] | Overview: {result['overview_text']} |')

| 1 | Title: [ Avatar ] | Overview: In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. |
| 2 | Title: [ Alien: Resurrection ] | Overview: Two hundred years after Lt. Ripley died, a group of scientists clone her, hoping to breed the ultimate weapon. But the new Ripley is full of surprises … as are the new aliens. Ripley must team with a band of smugglers to keep the creatures from reaching Earth. |
| 3 | Title: [ The Black Hole ] | Overview: The explorer craft U.S.S. Palomino is returning to Earth after a fruitless 18-month search for extra-terrestrial life when the crew comes upon a supposedly lost ship, the magnificent U.S.S. Cygnus, hovering near a black hole. The ship is controlled by Dr. Hans Reinhardt and his monstrous robot companion, Maximillian. But the initial wonderment and awe the Palomino crew feel for the ship and its resistance to the power of the b

In [8]:
# 2) 원하는 [줄거리] 입력 -> [영화] 추천
input_overview = "The explorer craft U.S.S. Palomino is returning to Earth after a fruitless 18-month search for extra-terrestrial life when the crew comes upon a supposedly lost ship, the magnificent U.S.S. Cygnus, hovering near a black hole. The ship is controlled by Dr. Hans Reinhardt and his monstrous robot companion, Maximillian. But the initial wonderment and awe the Palomino crew feel for the ship and its resistance to the power of the black hole turn to horror as they uncover Reinhardt's plans."
query_embed = movie_embedding_model.encode(input_overview).tolist()

top_n = 5

results = movie_collection.query(query_embeddings=query_embed, n_results=top_n)

for i, result in enumerate(results['metadatas'][0]):
    print(f'| {i+1} | Title: [ {result['title']} ] | Overview: {result['overview_text']} |')

| 1 | Title: [ The Black Hole ] | Overview: The explorer craft U.S.S. Palomino is returning to Earth after a fruitless 18-month search for extra-terrestrial life when the crew comes upon a supposedly lost ship, the magnificent U.S.S. Cygnus, hovering near a black hole. The ship is controlled by Dr. Hans Reinhardt and his monstrous robot companion, Maximillian. But the initial wonderment and awe the Palomino crew feel for the ship and its resistance to the power of the black hole turn to horror as they uncover Reinhardt's plans. |
| 2 | Title: [ Event Horizon ] | Overview: In the year 2047 a group of astronauts are sent to investigate and salvage the long lost starship "Event Horizon". The ship disappeared mysteriously 7 years before on its maiden voyage and with its return comes even more mystery as the crew of the "Lewis and Clark" discover the real truth behind its disappearance and something even more terrifying. |
| 3 | Title: [ Space Battleship Yamato ] | Overview: In 2199, five y